# Student Performance Prediction Using Machine Learning

## An Analysis of the Open University Learning Analytics Dataset (OULAD)

**Abstract:** This study investigates student performance prediction using the Open University Learning Analytics Dataset. We analyze behavioral patterns, demographic factors, and engagement metrics to build predictive models for student success. Through comprehensive exploratory data analysis, feature engineering, and machine learning techniques, we demonstrate the effectiveness of various algorithms in early identification of at-risk students.

**Keywords:** Educational Data Mining, Learning Analytics, Student Performance Prediction, Machine Learning, Virtual Learning Environment

---

## Table of Contents
1. [Dataset Overview](#1-dataset-overview)
2. [Data Loading and Preprocessing](#2-data-loading-and-preprocessing)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Feature Engineering](#4-feature-engineering)
5. [Model Training and Evaluation](#5-model-training-and-evaluation)
6. [Results and Discussion](#6-results-and-discussion)
7. [Conclusions](#7-conclusions)

---
## 1. Dataset Overview

The **Open University Learning Analytics Dataset (OULAD)** was released by the Open University UK and contains data about courses, students, and their interactions with the Virtual Learning Environment (VLE).

### Dataset Tables

| Table | Description | Key Columns |
|---|---|---|
| `studentInfo` | Demographic information and final results | id_student, gender, region, highest_education, imd_band, final_result |
| `courses` | Module and presentation metadata | code_module, code_presentation, module_presentation_length |
| `assessments` | Assessment details | id_assessment, assessment_type, date, weight |
| `studentAssessment` | Student scores per assessment | id_student, id_assessment, score, date_submitted |
| `vle` | Virtual Learning Environment activity catalogue | id_site, activity_type |
| `studentVle` | Student interactions with VLE resources | id_student, id_site, sum_click |
| `studentRegistration` | Registration and withdrawal dates | id_student, date_registration, date_unregistration |

### Target Variable
We binarise `final_result` into:
- **1 (Pass)** — Pass or Distinction
- **0 (At-Risk)** — Fail or Withdrawn

---
## 2. Data Loading and Preprocessing

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Use inline backend when running in notebook environments
try:
    get_ipython()
    %matplotlib inline
except NameError:
    pass

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='tab10')

# Add src to path
import sys
sys.path.insert(0, os.path.abspath('.'))

print('Environment ready.')

In [ ]:
from src.data_loader import OULADLoader

# Load data (falls back to synthetic OULAD-schema data if CSVs are not present)
loader = OULADLoader(data_dir='data', n_synthetic=5000, random_state=42)
loader.load()

student_info = loader.get_table('studentInfo')
master_df    = loader.build_master_dataset()

print(f'studentInfo shape : {student_info.shape}')
print(f'Master dataset    : {master_df.shape}')
student_info.head(3)

In [ ]:
# Basic statistics
print('=== Missing values (master dataset) ===')
print(master_df.isnull().sum()[master_df.isnull().sum() > 0])
print(f'\nClass distribution:')
print(master_df['final_result'].value_counts())
print(f'\nBinary outcome distribution:')
print(master_df['outcome'].value_counts(normalize=True).round(3))

---
## 3. Exploratory Data Analysis

### 3.1 Outcome Distribution

In [ ]:
from src.visualization import plot_outcome_distribution
fig = plot_outcome_distribution(student_info)
plt.show()

### 3.2 Demographic Analysis

In [ ]:
from src.visualization import plot_demographic_analysis
fig = plot_demographic_analysis(student_info)
plt.show()

**Key observations:**
- Higher education levels correlate strongly with Pass/Distinction outcomes.
- Students with no previous attempts tend to perform better.
- IMD (Index of Multiple Deprivation) band shows a socioeconomic gradient in outcomes.

### 3.3 VLE Engagement Patterns

In [ ]:
from src.visualization import plot_vle_engagement
fig = plot_vle_engagement(master_df)
plt.show()

**Key observations:**
- Students who pass show significantly higher VLE engagement (total clicks, unique sites visited, active days).
- Average clicks per day is a strong early predictor of academic success.

### 3.4 Assessment Score Analysis

In [ ]:
from src.visualization import plot_assessment_scores
fig = plot_assessment_scores(master_df)
plt.show()

### 3.5 Feature Correlation Matrix

In [ ]:
from src.feature_engineering import engineer_features
from src.visualization import plot_correlation_heatmap

X_raw, y = engineer_features(master_df)
fig = plot_correlation_heatmap(X_raw, y)
plt.show()

---
## 4. Feature Engineering

We construct the following feature groups:

| Feature Group | Features |
|---|---|
| **Demographic** | gender, highest_education (ordinal), imd_band (ordinal), age_band (ordinal), num_of_prev_attempts, studied_credits, disability |
| **VLE Engagement** | total_clicks, n_unique_sites, n_active_days, avg_clicks_per_day |
| **Assessment** | mean_score, min_score, max_score, n_submissions, n_late_submissions |
| **Registration** | date_registration, early_registration |

In [ ]:
from src.feature_engineering import engineer_features, get_feature_names

X, y = engineer_features(master_df)
feature_names = get_feature_names()

print(f'Feature matrix shape : {X.shape}')
print(f'Feature names        : {feature_names}')
X.describe().round(2)

---
## 5. Model Training and Evaluation

We train and compare the following classifiers:
- Logistic Regression (baseline)
- Decision Tree
- Random Forest
- Gradient Boosting
- Support Vector Machine (SVM)
- XGBoost
- LightGBM

Each model is wrapped in a scikit-learn Pipeline that applies median imputation and standard scaling before the classifier. We use 5-fold stratified cross-validation and report hold-out test performance.

In [ ]:
from src.models import ModelTrainer

trainer = ModelTrainer(random_state=42)
trainer.split(X, y)
trainer.train_all()

### 5.1 Model Comparison Table

In [ ]:
results_df = trainer.get_results_df()
results_df.style.background_gradient(subset=['Accuracy', 'F1 Score', 'ROC-AUC'], cmap='YlGn')

### 5.2 ROC Curves

In [ ]:
from src.evaluation import plot_roc_curves
fig = plot_roc_curves(trainer.results, trainer.X_test, trainer.y_test)
plt.show()

### 5.3 Precision-Recall Curves

In [ ]:
from src.evaluation import plot_precision_recall_curves
fig = plot_precision_recall_curves(trainer.results, trainer.X_test, trainer.y_test)
plt.show()

### 5.4 Model Comparison Bar Chart

In [ ]:
from src.evaluation import plot_model_comparison
fig = plot_model_comparison(results_df)
plt.show()

### 5.5 Best Model — Confusion Matrix and Classification Report

In [ ]:
from src.evaluation import plot_confusion_matrix, print_classification_report

best_name, best_result = trainer.best_model()
print(f'Best model: {best_name}  (ROC-AUC = {best_result.roc_auc:.4f})')

y_pred_best = best_result.pipeline.predict(trainer.X_test)

fig = plot_confusion_matrix(
    trainer.y_test, y_pred_best,
    title=f'Confusion Matrix — {best_name}',
)
plt.show()

print_classification_report(trainer.y_test, y_pred_best, model_name=best_name)

### 5.6 Feature Importance

In [ ]:
from src.evaluation import plot_feature_importance

# Find model with feature importances (tree-based models)
for name, result in trainer.results.items():
    if result.feature_importances is not None:
        fig = plot_feature_importance(
            feature_names, result.feature_importances,
            model_name=name, top_n=15,
        )
        plt.show()
        break

---
## 6. Results and Discussion

### Key Findings

1. **Predictive Power**: Ensemble methods (Random Forest, XGBoost, LightGBM) and Logistic Regression achieve high ROC-AUC scores, demonstrating that student success can be reliably predicted from available data.

2. **Most Predictive Features**:
   - **Assessment scores** (mean_score, min_score) are the strongest predictors.
   - **VLE engagement** metrics (total clicks, active days) are important behavioral predictors.
   - **Prior academic history** (num_of_prev_attempts) and **education level** contribute significant demographic signal.

3. **Early Warning Potential**: VLE engagement and early assessment data are available early in the course, enabling timely intervention for at-risk students.

4. **Class Imbalance**: The dataset has a moderate class imbalance (~55% Pass vs 45% At-Risk). All models use `class_weight='balanced'` to mitigate this.

### Implications for Educational Practice

- Institutions can deploy these models mid-course to flag students for additional support.
- VLE click patterns provide a non-intrusive and real-time signal for engagement monitoring.
- Demographic factors alone are insufficient — behavioral engagement features are essential.

In [ ]:
# Summary statistics
print('=== Final Results Summary ===')
print(results_df[['Model', 'Accuracy', 'F1 Score', 'ROC-AUC', 'CV Accuracy']].to_string(index=False))

---
## 7. Conclusions

This study demonstrates that **machine learning methods can effectively predict student performance** from the OULAD dataset. The key conclusions are:

1. **Behavioral engagement** (VLE clicks, active days) is a powerful predictor that can be measured in real time.
2. **Assessment scores** from formative assessments (TMAs) provide highly predictive signals.
3. **Ensemble methods** (Random Forest, XGBoost, LightGBM) consistently outperform simpler baselines.
4. **Early intervention systems** built on these models could significantly improve student outcomes.

### Future Work

- Incorporate time-series features from weekly VLE activity patterns.
- Explore deep learning approaches (LSTM, Transformers) on sequential click data.
- Build explainability tools (SHAP, LIME) for counsellor-facing interfaces.
- Validate on additional cohorts and module types to assess generalizability.

### References

- Kuzilek J., Hlosta M., Zdrahal Z. (2017). *Open University Learning Analytics Dataset*. Scientific Data 4, 170171. https://doi.org/10.1038/sdata.2017.171
- Romero C., Ventura S. (2010). *Educational data mining: A review of the state of the art*. IEEE Transactions on Systems, Man, and Cybernetics, Part C, 40(6), 601–618.